# 37 -- Section 01e Steps 1-6: b switched from movie popularity to user rating habit

New definition: b = (mu_u - 0.5) / 4.5, where mu_u is the worked-example
user's mean rating over their TRAINING ratings only. b depends only on the
user, not the movie. In both channels, b is added to every movie in this
user's rated movie list (channel + : a_i + b, channel - : a_i - b);
unrated movies stay 0 in both channels.

Reuses the exact same user, movie (Rumble in the Bronx), hidden unit h1,
seeds, starting weights/biases, learning rate and random-draw procedure as
notebooks 31, 32, 33, 34, 35 and 36. The 01c scenario (b = 0) is unchanged
from those notebooks; only the 01e scenario is recomputed here. Does not
touch the HTML.

## Part 0 -- Setup: scan notebooks/outputs, pick this notebook's number

In [1]:
from pathlib import Path
import json
import re
import math

import pandas as pd
import torch

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
notebooks_dir = root / "notebooks"
data_dir = root / "data"
out_dir = root / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)

nb_paths = sorted(p for p in notebooks_dir.glob("*.ipynb") if "rating_habit_b" not in p.stem)
num_re = re.compile(r"^(\d+)")
numbered = sorted(
    (int(m.group(1)), p) for p in nb_paths if (m := num_re.match(p.stem))
)
highest_num, highest_path = numbered[-1]
this_num = highest_num + 1
this_name = f"{this_num}_step1to6_rating_habit_b.ipynb"
print(f"highest existing notebook: {highest_num:>3d}  ({highest_path.name})")
print(f"this notebook is:          {this_num:>3d}  ({this_name})")

highest existing notebook:  36  (36_step6_training.ipynb)
this notebook is:           37  (37_step1to6_rating_habit_b.ipynb)


## Part 1 -- Reuse the exact setup from notebooks 31/32/35/36

In [2]:
train_path = data_dir / "train_hyperbolic.csv"
test_path = data_dir / "test_hyperbolic.csv"
results31_path = out_dir / "31_step8_results.json"
for p in (train_path, test_path, results31_path):
    assert p.exists(), f"Missing {p}"

train_hyp = pd.read_csv(train_path)
test_hyp = pd.read_csv(test_path)
assert len(train_hyp) == 492055 and len(test_hyp) == 123474

results31 = json.loads(results31_path.read_text())
REAL_SEED = results31["seeds"]["plus_seed_reused"]
MINUS_SEED = results31["seeds"]["minus_seed_trained"]
N_HIDDEN = int(results31["architecture_match"]["128 hidden units"].split("=")[1])
LR = results31["worked_example_plus"]["lr"]
H1_IDX = 0

all_users = sorted(pd.concat([train_hyp["userId"], test_hyp["userId"]]).unique())
all_movies = sorted(pd.concat([train_hyp["movieId"], test_hyp["movieId"]]).unique())
n_movies = len(all_movies)
movie_to_col = {m: j for j, m in enumerate(all_movies)}

movies = pd.read_csv(data_dir / "movie.csv")
RUMBLE_ID = int(movies[movies["title"].str.contains("Rumble in the Bronx", case=False, na=False)].iloc[0]["movieId"])
RUMBLE_TITLE = movies[movies["movieId"] == RUMBLE_ID].iloc[0]["title"]
rumble_col = movie_to_col[RUMBLE_ID]
MEDIAN_USER = results31["worked_example_plus"]["user"]
print(f"movie: {RUMBLE_TITLE} (column={rumble_col})  user: {MEDIAN_USER}")

user_rows = train_hyp[train_hyp["userId"] == MEDIAN_USER]
v0_full = torch.zeros(n_movies)   # a_i only, one per rated movie -- the 01c scenario, unchanged
mask0_full = torch.zeros(n_movies)
for row in user_rows.itertuples():
    v0_full[movie_to_col[row.movieId]] = row.a
    mask0_full[movie_to_col[row.movieId]] = 1.0
mask0b = mask0_full.unsqueeze(0)
rated_idx = mask0_full.nonzero(as_tuple=True)[0]
a_rumble = v0_full[rumble_col].item()
assert abs(a_rumble - results31["worked_example_plus"]["v0"]) < 1e-6

g_init_p = torch.Generator().manual_seed(REAL_SEED)
W_plus_init = torch.randn(n_movies, N_HIDDEN, generator=g_init_p) * 0.01
hbias_plus_init = torch.zeros(N_HIDDEN)
vbias_plus_init = torch.zeros(n_movies)

g_init_m = torch.Generator().manual_seed(MINUS_SEED)
W_minus_init = torch.randn(n_movies, N_HIDDEN, generator=g_init_m) * 0.01
hbias_minus_init = torch.zeros(N_HIDDEN)
vbias_minus_init = torch.zeros(n_movies)

w_plus_rumble = W_plus_init[rumble_col, H1_IDX].item()
w_minus_rumble = W_minus_init[rumble_col, H1_IDX].item()
print(f"a[Rumble]={a_rumble:.6f}  rated movie count={len(rated_idx)}")

movie: Rumble in the Bronx (Hont faan kui) (1995) (column=27)  user: 24127
a[Rumble]=0.777778  rated movie count=530


## Part 2 -- Step 1: n, mu_u, b, channel + / channel - inputs for Rumble

In [3]:
n_ratings = len(user_rows)
sum_ratings = user_rows["rating"].sum()
mu_u = sum_ratings / n_ratings
b_val = (mu_u - 0.5) / 4.5

assert n_ratings == int(mask0_full.sum().item()) == len(rated_idx)

channel_plus_rumble_01c = a_rumble
channel_minus_rumble_01c = a_rumble
channel_plus_rumble_01e = a_rumble + b_val
channel_minus_rumble_01e = a_rumble - b_val

print(f"n = {n_ratings}")
print(f"sum of ratings = {sum_ratings}")
print(f"mu_u = {mu_u:.6f}")
print(f"b = {b_val:.6f}")
print(f"channel+ (Rumble): 01c={channel_plus_rumble_01c:.4f}  01e={channel_plus_rumble_01e:.4f}")
print(f"channel- (Rumble): 01c={channel_minus_rumble_01c:.4f}  01e={channel_minus_rumble_01e:.4f}")

# general reachability check, independent of this user
assert abs((2.5 - 0.5) / 4.5 - 0.4444) < 5e-4
assert abs((4.0 - 0.5) / 4.5 - 0.7778) < 5e-4
assert ((2.75 - 0.5) / 4.5) < 0.5 + 1e-9 and ((2.751 - 0.5) / 4.5) >= 0.5 - 1e-3
print("boundary identities (strict/generous/reachability thresholds) confirmed in code.")

n = 530
sum of ratings = 1831.5
mu_u = 3.455660
b = 0.656813
channel+ (Rumble): 01c=0.7778  01e=1.4346
channel- (Rumble): 01c=0.7778  01e=0.1210
boundary identities (strict/generous/reachability thresholds) confirmed in code.


## Part 3 -- Step 2: sums of w+ / w- over the rated movie list R, x+ and x- for 01e

b is added to every rated entry of v, so the change in x is exactly b times
the sum of that channel's weight column over every rated movie (linearity
of the dot product), cross-checked against a full recomputation from the
modified visible vectors.

In [4]:
sum_w_plus_R = W_plus_init[rated_idx, H1_IDX].sum().item()
sum_w_minus_R = W_minus_init[rated_idx, H1_IDX].sum().item()
R_size = len(rated_idx)

x_plus_01c = (v0_full @ W_plus_init + hbias_plus_init)[H1_IDX].item()
x_minus_01c = (v0_full @ W_minus_init + hbias_minus_init)[H1_IDX].item()

x_plus_01e_shortcut = x_plus_01c + b_val * sum_w_plus_R
x_minus_01e_shortcut = x_minus_01c - b_val * sum_w_minus_R

# full recomputation for cross-check: v_i = a_i + b for every rated movie, 0 elsewhere
v_plus_01e_full = torch.where(mask0_full.bool(), v0_full + b_val, torch.zeros(n_movies))
v_minus_01e_full = torch.where(mask0_full.bool(), v0_full - b_val, torch.zeros(n_movies))
x_plus_01e_full = (v_plus_01e_full @ W_plus_init + hbias_plus_init)[H1_IDX].item()
x_minus_01e_full = (v_minus_01e_full @ W_minus_init + hbias_minus_init)[H1_IDX].item()

assert abs(x_plus_01e_shortcut - x_plus_01e_full) < 1e-4
assert abs(x_minus_01e_shortcut - x_minus_01e_full) < 1e-4
x_plus_01e = x_plus_01e_full
x_minus_01e = x_minus_01e_full

print(f"|R| = {R_size}")
print(f"sum w+ over R (h1) = {sum_w_plus_R:.6f}")
print(f"sum w- over R (h1) = {sum_w_minus_R:.6f}")
print(f"x+_01c = {x_plus_01c:.6f}  x+_01e = {x_plus_01e:.6f}  (shortcut matches full recompute: {abs(x_plus_01e_shortcut-x_plus_01e_full)<1e-4})")
print(f"x-_01c = {x_minus_01c:.6f}  x-_01e = {x_minus_01e:.6f}  (shortcut matches full recompute: {abs(x_minus_01e_shortcut-x_minus_01e_full)<1e-4})")

|R| = 530
sum w+ over R (h1) = -0.332340
sum w- over R (h1) = -0.325376
x+_01c = -0.211115  x+_01e = -0.429400  (shortcut matches full recompute: True)
x-_01c = -0.205367  x-_01e = 0.008345  (shortcut matches full recompute: True)


## Part 4 -- Step 3: q+, q-, real part and k part for 01e

In [5]:
def sigmoid(z):
    return 1.0 / (1.0 + math.exp(-z))

q_plus_01c = sigmoid(x_plus_01c)
q_minus_01c = sigmoid(x_minus_01c)
q_plus_01e = sigmoid(x_plus_01e)
q_minus_01e = sigmoid(x_minus_01e)

q_real_01c = (q_plus_01c + q_minus_01c) / 2
q_k_01c = (q_plus_01c - q_minus_01c) / 2
q_real_01e = (q_plus_01e + q_minus_01e) / 2
q_k_01e = (q_plus_01e - q_minus_01e) / 2

print(f"q+ : 01c={q_plus_01c:.6f}  01e={q_plus_01e:.6f}")
print(f"q- : 01c={q_minus_01c:.6f}  01e={q_minus_01e:.6f}")
print(f"Re(q) : 01c={q_real_01c:.6f}  01e={q_real_01e:.6f}")
print(f"k(q)  : 01c={q_k_01c:.6f}  01e={q_k_01e:.6f}")

q+ : 01c=0.447416  01e=0.394270
q- : 01c=0.448838  01e=0.502086
Re(q) : 01c=0.448127  01e=0.448178
k(q)  : 01c=-0.000711  01e=-0.053908


## Part 5 -- Step 4: the four joint-state probabilities (01e only)

In [6]:
P_00 = (1 - q_plus_01e) * (1 - q_minus_01e)
P_10 = q_plus_01e * (1 - q_minus_01e)
P_01 = (1 - q_plus_01e) * q_minus_01e
P_11 = q_plus_01e * q_minus_01e
total = P_00 + P_10 + P_01 + P_11
assert abs(total - 1.0) < 1e-9
print(f"P(0,0)={P_00:.6f}  P(1,0)={P_10:.6f}  P(0,1)={P_01:.6f}  P(1,1)={P_11:.6f}  sum={total:.6f}")

P(0,0)=0.301602  P(1,0)=0.196312  P(0,1)=0.304129  P(1,1)=0.197957  sum=1.000000


## Part 6 -- Step 5: full 128-unit sampling, bit differences vs 01c, reconstruction

The 01c hidden sample is recomputed here from the unmodified `v0_full` with
the same seeds, to compare directly against the 01e sample built from the
new rating-habit `v`.

In [7]:
def cd1_round_trip(v_vec, W, hbias, vbias, mask, seed):
    vb = v_vec.unsqueeze(0)
    h0_prob = torch.sigmoid(vb @ W + hbias)
    g = torch.Generator().manual_seed(seed)
    s = torch.bernoulli(h0_prob, generator=g)
    v1_prob = torch.sigmoid(s @ W.t() + vbias) * mask
    h1_prob = torch.sigmoid(v1_prob @ W + hbias)
    return h0_prob, s, v1_prob, h1_prob

h0_plus_01c, s_plus_01c, v1_plus_01c, h1_plus_01c = cd1_round_trip(v0_full, W_plus_init, hbias_plus_init, vbias_plus_init, mask0b, REAL_SEED)
h0_minus_01c, s_minus_01c, v1_minus_01c, h1_minus_01c = cd1_round_trip(v0_full, W_minus_init, hbias_minus_init, vbias_minus_init, mask0b, MINUS_SEED)
h0_plus_01e, s_plus_01e, v1_plus_01e, h1_plus_01e = cd1_round_trip(v_plus_01e_full, W_plus_init, hbias_plus_init, vbias_plus_init, mask0b, REAL_SEED)
h0_minus_01e, s_minus_01e, v1_minus_01e, h1_minus_01e = cd1_round_trip(v_minus_01e_full, W_minus_init, hbias_minus_init, vbias_minus_init, mask0b, MINUS_SEED)

bits_differ_plus = int((s_plus_01c != s_plus_01e).sum().item())
bits_differ_minus = int((s_minus_01c != s_minus_01e).sum().item())
print(f"plus channel bits differing (01c vs 01e, out of 128): {bits_differ_plus}")
print(f"minus channel bits differing (01c vs 01e, out of 128): {bits_differ_minus}")

vhat_plus_01c = v1_plus_01c[0, rumble_col].item()
vhat_minus_01c = v1_minus_01c[0, rumble_col].item()
vhat_plus_01e = v1_plus_01e[0, rumble_col].item()
vhat_minus_01e = v1_minus_01e[0, rumble_col].item()

assert abs(vhat_plus_01c - results31["worked_example_plus"]["v1"]) < 5e-4
assert abs(vhat_minus_01c - results31["worked_example_minus"]["v1"]) < 5e-4
print("01c v-hat values match notebook 31. OK.")

def combine(vp, vm):
    ahat = (vp + vm) / 2
    bhat = (vp - vm) / 2
    rhat = 4.5 * ahat + 0.5
    return ahat, bhat, rhat

ahat_01c, bhat_01c, rhat_01c = combine(vhat_plus_01c, vhat_minus_01c)
ahat_01e, bhat_01e, rhat_01e = combine(vhat_plus_01e, vhat_minus_01e)

for bhat in (bhat_01c, bhat_01e):
    assert -0.5 < bhat < 0.5
print(f"01c: v+hat={vhat_plus_01c:.6f} v-hat={vhat_minus_01c:.6f} a-hat={ahat_01c:.6f} r-hat={rhat_01c:.6f} b-hat={bhat_01c:.6f}")
print(f"01e: v+hat={vhat_plus_01e:.6f} v-hat={vhat_minus_01e:.6f} a-hat={ahat_01e:.6f} r-hat={rhat_01e:.6f} b-hat={bhat_01e:.6f}")

plus channel bits differing (01c vs 01e, out of 128): 4
minus channel bits differing (01c vs 01e, out of 128): 3
01c v-hat values match notebook 31. OK.
01c: v+hat=0.488463 v-hat=0.463210 a-hat=0.475837 r-hat=2.641264 b-hat=0.012627
01e: v+hat=0.491774 v-hat=0.456431 a-hat=0.474102 r-hat=2.633459 b-hat=0.017672


## Part 7 -- Step 6: CD-1 update dW+, dW-, dB for 01e

In [8]:
q0_plus_01e = h0_plus_01e[0, H1_IDX].item()
q0_minus_01e = h0_minus_01e[0, H1_IDX].item()
q1_plus_01e = h1_plus_01e[0, H1_IDX].item()
q1_minus_01e = h1_minus_01e[0, H1_IDX].item()

assert abs(q0_plus_01e - q_plus_01e) < 1e-4
assert abs(q0_minus_01e - q_minus_01e) < 1e-4

q0_plus_01c = h0_plus_01c[0, H1_IDX].item()
q0_minus_01c = h0_minus_01c[0, H1_IDX].item()
q1_plus_01c = h1_plus_01c[0, H1_IDX].item()
q1_minus_01c = h1_minus_01c[0, H1_IDX].item()

dW_plus_01c = LR * (a_rumble * q0_plus_01c - vhat_plus_01c * q1_plus_01c)
dW_minus_01c = LR * (a_rumble * q0_minus_01c - vhat_minus_01c * q1_minus_01c)
dW_plus_01e = LR * (channel_plus_rumble_01e * q0_plus_01e - vhat_plus_01e * q1_plus_01e)
dW_minus_01e = LR * (channel_minus_rumble_01e * q0_minus_01e - vhat_minus_01e * q1_minus_01e)

dB_01c = (dW_plus_01c - dW_minus_01c) / 2
dB_01e = (dW_plus_01e - dW_minus_01e) / 2

assert abs(dW_plus_01c - 0.0062) < 5e-4
assert abs(dW_minus_01c - 0.0067) < 5e-4
print("01c column matches notebook 36 / the page's Step 8 table. OK.")
print(f"dW+ : 01c={dW_plus_01c:.6f}  01e={dW_plus_01e:.6f}")
print(f"dW- : 01c={dW_minus_01c:.6f}  01e={dW_minus_01e:.6f}")
print(f"dB  : 01c={dB_01c:.6f}  01e={dB_01e:.6f}")

01c column matches notebook 36 / the page's Step 8 table. OK.
dW+ : 01c=0.006163  01e=0.016973
dW- : 01c=0.006682  01e=-0.007585
dB  : 01c=-0.000259  01e=0.012279


## Part 8 -- Save for section 01e (Steps 1-6, rating-habit b)

In [9]:
results = {
    "notebook": this_name,
    "reused_from": "31_step8_minus_channel.ipynb, 32_step2_hidden_input.ipynb, 33_step3_hidden_probability.ipynb, 34_step4_hidden_state.ipynb, 35_step5_visible_reconstruction.ipynb, 36_step6_training.ipynb",
    "movie": RUMBLE_TITLE,
    "user": MEDIAN_USER,
    "hidden_unit": "h1 (index 0)",
    "step1": {
        "n": n_ratings,
        "sum_ratings": float(sum_ratings),
        "mu_u": mu_u,
        "b": b_val,
        "channel_plus_rumble": {"01c": channel_plus_rumble_01c, "01e": channel_plus_rumble_01e},
        "channel_minus_rumble": {"01c": channel_minus_rumble_01c, "01e": channel_minus_rumble_01e},
    },
    "step2": {
        "R_size": R_size,
        "sum_w_plus_R": sum_w_plus_R,
        "sum_w_minus_R": sum_w_minus_R,
        "x_plus": {"01c": x_plus_01c, "01e": x_plus_01e},
        "x_minus": {"01c": x_minus_01c, "01e": x_minus_01e},
    },
    "step3": {
        "q_plus": {"01c": q_plus_01c, "01e": q_plus_01e},
        "q_minus": {"01c": q_minus_01c, "01e": q_minus_01e},
        "q_real": {"01c": q_real_01c, "01e": q_real_01e},
        "q_k": {"01c": q_k_01c, "01e": q_k_01e},
    },
    "step4": {"P_00": P_00, "P_10": P_10, "P_01": P_01, "P_11": P_11, "sum": total},
    "step5": {
        "bits_differ_plus": bits_differ_plus,
        "bits_differ_minus": bits_differ_minus,
        "vhat_plus": {"01c": vhat_plus_01c, "01e": vhat_plus_01e},
        "vhat_minus": {"01c": vhat_minus_01c, "01e": vhat_minus_01e},
        "ahat": {"01c": ahat_01c, "01e": ahat_01e},
        "rhat": {"01c": rhat_01c, "01e": rhat_01e},
        "bhat": {"01c": bhat_01c, "01e": bhat_01e},
    },
    "step6": {
        "q0_plus": {"01c": q0_plus_01c, "01e": q0_plus_01e},
        "q0_minus": {"01c": q0_minus_01c, "01e": q0_minus_01e},
        "q1_plus": {"01c": q1_plus_01c, "01e": q1_plus_01e},
        "q1_minus": {"01c": q1_minus_01c, "01e": q1_minus_01e},
        "dW_plus": {"01c": dW_plus_01c, "01e": dW_plus_01e},
        "dW_minus": {"01c": dW_minus_01c, "01e": dW_minus_01e},
        "dB": {"01c": dB_01c, "01e": dB_01e},
    },
}
results_path = out_dir / "37_step1to6_rating_habit_b_results.json"
with results_path.open("w") as f:
    json.dump(results, f, indent=2)
print(f"saved {results_path}")
print(json.dumps(results, indent=2))

saved /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/outputs/37_step1to6_rating_habit_b_results.json
{
  "notebook": "37_step1to6_rating_habit_b.ipynb",
  "reused_from": "31_step8_minus_channel.ipynb, 32_step2_hidden_input.ipynb, 33_step3_hidden_probability.ipynb, 34_step4_hidden_state.ipynb, 35_step5_visible_reconstruction.ipynb, 36_step6_training.ipynb",
  "movie": "Rumble in the Bronx (Hont faan kui) (1995)",
  "user": 24127,
  "hidden_unit": "h1 (index 0)",
  "step1": {
    "n": 530,
    "sum_ratings": 1831.5,
    "mu_u": 3.4556603773584906,
    "b": 0.6568134171907757,
    "channel_plus_rumble": {
      "01c": 0.7777777910232544,
      "01e": 1.4345912082140302
    },
    "channel_minus_rumble": {
      "01c": 0.7777777910232544,
      "01e": 0.12096437383247871
    }
  },
  "step2": {
    "R_size": 530,
    "sum_w_plus_R": -0.33233973383903503,
    "sum_w_minus_R": -0.3253764510154724,
    "x_plus": {
      "01c": -0.21111512184143066,
      "01e": -0.429400235414505
  